# Airfoil Self-Noise Regression

Refined notebook for regression using the UCI Airfoil Self-Noise dataset. It follows the assignment requirements: data loading, preprocessing, EDA, training (Linear Regression, Decision Tree Regressor, SVR, MLPRegressor), evaluation and visualizations.

## 1. Imports and utilities

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42
sns.set_style('whitegrid')


## 2. Data download and loading
We load the Airfoil dataset directly from the UCI repository. If you run this offline, download the file and place it in the notebook folder.

In [ ]:
# The UCI file uses whitespace separators and has no header
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00291/airfoil_self_noise.dat'
cols = ['Frequency', 'AngleAttack', 'ChordLength', 'FreeStreamVelocity', 'SuctionSideDisplacementThickness', 'SoundPressureLevel']

df = pd.read_csv(url, sep='\t', header=None, names=cols, engine='python')
print('Loaded shape:', df.shape)
df.head()

## 3. Preprocessing
Scale input features. No missing values expected for this dataset.

In [ ]:
X = df.drop(columns=['SoundPressureLevel'])
y = df['SoundPressureLevel']

# Check for missing values
print('Missing values per column:')
print(df.isnull().sum())
print('\nBasic statistics:')
print(df.describe())

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)


## 4. Exploratory analysis
Plot relationships and distribution of target.

In [ ]:
# Create a comprehensive EDA with multiple plots
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Plot 1: Frequency vs SoundPressureLevel
axes[0, 0].scatter(df['Frequency'], y, s=12, alpha=0.6)
axes[0, 0].set_xlabel('Frequency')
axes[0, 0].set_ylabel('SoundPressureLevel')
axes[0, 0].set_title('Frequency vs SoundPressureLevel')

# Plot 2: AngleAttack vs SoundPressureLevel
axes[0, 1].scatter(df['AngleAttack'], y, s=12, alpha=0.6, color='orange')
axes[0, 1].set_xlabel('AngleAttack')
axes[0, 1].set_ylabel('SoundPressureLevel')
axes[0, 1].set_title('AngleAttack vs SoundPressureLevel')

# Plot 3: ChordLength vs SoundPressureLevel
axes[0, 2].scatter(df['ChordLength'], y, s=12, alpha=0.6, color='green')
axes[0, 2].set_xlabel('ChordLength')
axes[0, 2].set_ylabel('SoundPressureLevel')
axes[0, 2].set_title('ChordLength vs SoundPressureLevel')

# Plot 4: Histogram of target
axes[1, 0].hist(y, bins=30, edgecolor='black', alpha=0.7)
axes[1, 0].set_title('Distribution of Sound Pressure Level')
axes[1, 0].set_xlabel('SoundPressureLevel')
axes[1, 0].set_ylabel('Frequency')

# Plot 5: Boxplot of target
axes[1, 1].boxplot(y)
axes[1, 1].set_title('Boxplot of Sound Pressure Level')
axes[1, 1].set_ylabel('SoundPressureLevel')

# Plot 6: Correlation heatmap
corr_matrix = df.corr()
im = axes[1, 2].imshow(corr_matrix, cmap='coolwarm', aspect='auto', vmin=-1, vmax=1)
axes[1, 2].set_xticks(range(len(corr_matrix.columns)))
axes[1, 2].set_yticks(range(len(corr_matrix.columns)))
axes[1, 2].set_xticklabels(corr_matrix.columns, rotation=45, ha='right', fontsize=8)
axes[1, 2].set_yticklabels(corr_matrix.columns, fontsize=8)
axes[1, 2].set_title('Correlation Heatmap')
plt.colorbar(im, ax=axes[1, 2])

plt.tight_layout()
plt.show()

# Feature correlations with target
print('\nCorrelation with target (SoundPressureLevel):')
print(df.corr()['SoundPressureLevel'].sort_values(ascending=False))


## 5. Split and model runner
Function to run regressors and compute metrics: MAE, MSE, RMSE, R2.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=RANDOM_STATE)
print('Train shape:', X_train.shape, 'Test shape:', X_test.shape)

def run_regressor(name, model, X_train, y_train, X_test, y_test, do_grid=False, param_grid=None):
    if do_grid and (param_grid is not None):
        gs = GridSearchCV(model, param_grid, cv=5, n_jobs=-1, scoring='r2', verbose=1)
        gs.fit(X_train, y_train)
        fitted = gs.best_estimator_
        extra = {'best_params': gs.best_params_, 'cv_best_score': gs.best_score_}
        print(f"\n{name} - Best parameters: {gs.best_params_}")
        print(f"{name} - Best CV R2 score: {gs.best_score_:.4f}")
    else:
        fitted = model.fit(X_train, y_train)
        extra = {}
    
    # Cross-validation scores
    cv_scores = cross_val_score(fitted, X_train, y_train, cv=5, scoring='r2')
    extra['cv_scores'] = cv_scores
    extra['cv_mean'] = cv_scores.mean()
    extra['cv_std'] = cv_scores.std()
    
    preds = fitted.predict(X_test)
    mse = mean_squared_error(y_test, preds)
    mae = mean_absolute_error(y_test, preds)
    
    # Calculate MAPE only for non-zero values to avoid division by zero
    mask = y_test != 0
    mape = mean_absolute_percentage_error(y_test[mask], preds[mask]) if mask.sum() > 0 else np.nan
    
    metrics = {
        'MAE': mae,
        'MSE': mse,
        'RMSE': np.sqrt(mse),
        'R2': r2_score(y_test, preds),
        'MAPE': mape,
        'predictions': preds
    }
    res = {'name': name, 'estimator': fitted, 'metrics': metrics}
    res.update(extra)
    return res


## 6. Models: Linear Regression, Decision Tree, SVR, MLPRegressor

In [ ]:
# Define models with hyperparameter grids
models_config = {
    'Linear Regression': {
        'model': LinearRegression(),
        'grid': None
    },
    'Decision Tree': {
        'model': DecisionTreeRegressor(random_state=RANDOM_STATE),
        'grid': {
            'max_depth': [5, 10, 15, 20, None],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 4]
        }
    },
    'Random Forest': {
        'model': RandomForestRegressor(random_state=RANDOM_STATE),
        'grid': {
            'n_estimators': [50, 100, 200],
            'max_depth': [10, 20, None],
            'min_samples_split': [2, 5]
        }
    },
    'SVR': {
        'model': SVR(),
        'grid': {
            'C': [10, 100, 200, 300],
            'gamma': ['scale', 'auto', 0.001, 0.01],
            'kernel': ['rbf', 'linear']
        }
    },
    'Gradient Boosting': {
        'model': GradientBoostingRegressor(
            random_state=RANDOM_STATE,
            n_iter_no_change=10,
            validation_fraction=0.1
            ),
        'grid': {
            'n_estimators': [50, 100, 200, 300],
            'learning_rate': [0.005, 0.01, 0.1, 0.2],
            'max_depth': [3, 5, 7, 9]
        }
    },
    'MLPRegressor': {
        'model': MLPRegressor(
            max_iter=15000, 
            random_state=RANDOM_STATE,
            early_stopping=True,
            n_iter_no_change=20,
            validation_fraction=0.1
            ),
        'grid': {
            'hidden_layer_sizes': [(150, 50),(150, 100), (50, 50, 50), (50, 100, 50)],
            'alpha': [0.005, 0.01, 0.05, 0.1],
            'learning_rate': ['constant', 'adaptive'],
            'activation': ['relu', 'tanh', 'logistic'],
        }
    }
}

results = {}
for name, config in models_config.items():
    print(f"\n{'='*60}")
    print(f"Training {name}...")
    print('='*60)
    res = run_regressor(
        name, 
        config['model'], 
        X_train, y_train, 
        X_test, y_test,
        do_grid=(config['grid'] is not None),
        param_grid=config['grid']
    )
    results[name] = res
    print(f"{name}: R2={res['metrics']['R2']:.4f}, RMSE={res['metrics']['RMSE']:.4f}, MAE={res['metrics']['MAE']:.4f}")
    if 'cv_mean' in res:
        print(f"{name}: CV R2={res['cv_mean']:.4f} (+/- {res['cv_std']:.4f})")


In [ ]:
# Create comparison dataframe
comparison_data = []
for name, res in results.items():
    row = {
        'Model': name,
        'R2': res['metrics']['R2'],
        'RMSE': res['metrics']['RMSE'],
        'MAE': res['metrics']['MAE'],
        'MAPE': res['metrics']['MAPE'],
    }
    if 'cv_mean' in res:
        row['CV_R2_Mean'] = res['cv_mean']
        row['CV_R2_Std'] = res['cv_std']
    comparison_data.append(row)

comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.sort_values('R2', ascending=False)
print('\nModel Performance Comparison:')
print('='*80)
print(comparison_df.to_string(index=False))
print('='*80)

# Visualize metrics comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# R2 Score comparison
axes[0, 0].barh(comparison_df['Model'], comparison_df['R2'], color='skyblue')
axes[0, 0].set_xlabel('R² Score')
axes[0, 0].set_title('R² Score Comparison')
axes[0, 0].set_xlim([0, 1])

# RMSE comparison
axes[0, 1].barh(comparison_df['Model'], comparison_df['RMSE'], color='coral')
axes[0, 1].set_xlabel('RMSE')
axes[0, 1].set_title('RMSE Comparison (Lower is Better)')

# MAE comparison
axes[1, 0].barh(comparison_df['Model'], comparison_df['MAE'], color='lightgreen')
axes[1, 0].set_xlabel('MAE')
axes[1, 0].set_title('MAE Comparison (Lower is Better)')

# MAPE comparison
axes[1, 1].barh(comparison_df['Model'], comparison_df['MAPE'], color='plum')
axes[1, 1].set_xlabel('MAPE (%)')
axes[1, 1].set_title('MAPE Comparison (Lower is Better)')

plt.tight_layout()
plt.show()

### 6.1 Model Performance Comparison

### 6.2 Detailed Analysis of Best Model

In [ ]:
# Compare top 3 models
top_3_names = sorted(results.keys(), key=lambda n: results[n]['metrics']['R2'], reverse=True)[:3]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, name in enumerate(top_3_names):
    preds = results[name]['metrics']['predictions']
    axes[idx].scatter(y_test, preds, s=20, alpha=0.6)
    axes[idx].set_xlabel('True SoundPressureLevel')
    axes[idx].set_ylabel('Predicted SoundPressureLevel')
    axes[idx].set_title(f'{name}\nR²={results[name]["metrics"]["R2"]:.4f}')
    mn = min(y_test.min(), preds.min())
    mx = max(y_test.max(), preds.max())
    axes[idx].plot([mn, mx], [mn, mx], 'r--', lw=2)
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# Find best by R2
best_name = max(results.keys(), key=lambda n: results[n]['metrics']['R2'])
best = results[best_name]
print('Best model:', best_name)
print(f"R² Score: {best['metrics']['R2']:.4f}")
print(f"RMSE: {best['metrics']['RMSE']:.4f}")
print(f"MAE: {best['metrics']['MAE']:.4f}")
if 'best_params' in best:
    print(f"Best Parameters: {best['best_params']}")

preds = best['metrics']['predictions']

# Create comprehensive visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Real vs Predicted
axes[0, 0].scatter(y_test, preds, s=20, alpha=0.6)
axes[0, 0].set_xlabel('True SoundPressureLevel')
axes[0, 0].set_ylabel('Predicted SoundPressureLevel')
axes[0, 0].set_title(f'Real vs Predicted ({best_name})')
mn = min(y_test.min(), preds.min())
mx = max(y_test.max(), preds.max())
axes[0, 0].plot([mn, mx], [mn, mx], 'r--', lw=2, label='Perfect Prediction')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Residuals plot
residuals = y_test - preds
axes[0, 1].scatter(preds, residuals, s=20, alpha=0.6)
axes[0, 1].axhline(y=0, color='r', linestyle='--', lw=2)
axes[0, 1].set_xlabel('Predicted SoundPressureLevel')
axes[0, 1].set_ylabel('Residuals')
axes[0, 1].set_title(f'Residual Plot ({best_name})')
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Residuals distribution
axes[1, 0].hist(residuals, bins=30, edgecolor='black', alpha=0.7)
axes[1, 0].set_xlabel('Residuals')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('Distribution of Residuals')
axes[1, 0].axvline(x=0, color='r', linestyle='--', lw=2)
axes[1, 0].grid(True, alpha=0.3)

# Plot 4: Error distribution
errors = np.abs(residuals)
axes[1, 1].hist(errors, bins=30, edgecolor='black', alpha=0.7, color='orange')
axes[1, 1].set_xlabel('Absolute Error')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('Distribution of Absolute Errors')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Additional statistics
print(f"\nResidual Statistics:")
print(f"Mean Residual: {residuals.mean():.4f}")
print(f"Std Residual: {residuals.std():.4f}")
print(f"Min Residual: {residuals.min():.4f}")
print(f"Max Residual: {residuals.max():.4f}")


### 6.4 Cross-Validation Performance Analysis

In [ ]:
# Cross-validation comparison for all models
cv_data = []
for name, res in results.items():
    if 'cv_scores' in res:
        cv_data.append({
            'Model': name,
            'CV_Mean': res['cv_mean'],
            'CV_Std': res['cv_std'],
            'Test_R2': res['metrics']['R2']
        })

cv_df = pd.DataFrame(cv_data).sort_values('CV_Mean', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Box plot of cross-validation scores
cv_scores_list = [results[name]['cv_scores'] for name in cv_df['Model']]
axes[0].boxplot(cv_scores_list, labels=cv_df['Model'])
axes[0].set_ylabel('R² Score')
axes[0].set_title('Cross-Validation Score Distribution')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(True, alpha=0.3)

# CV vs Test performance
x_pos = np.arange(len(cv_df))
width = 0.35
axes[1].bar(x_pos - width/2, cv_df['CV_Mean'], width, label='CV Mean R²', alpha=0.8, yerr=cv_df['CV_Std'], capsize=5)
axes[1].bar(x_pos + width/2, cv_df['Test_R2'], width, label='Test R²', alpha=0.8)
axes[1].set_xlabel('Model')
axes[1].set_ylabel('R² Score')
axes[1].set_title('Cross-Validation vs Test Performance')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(cv_df['Model'], rotation=45, ha='right')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('\nCross-Validation Summary:')
print(cv_df.to_string(index=False))

### 6.5 Feature Importance Analysis (Tree-based Models)

In [ ]:
# Analyze feature importance for tree-based models
tree_models = ['Decision Tree', 'Random Forest', 'Gradient Boosting']
available_tree_models = [name for name in tree_models if name in results]

if available_tree_models:
    n_models = len(available_tree_models)
    fig, axes = plt.subplots(1, n_models, figsize=(6*n_models, 5))
    if n_models == 1:
        axes = [axes]
    
    for idx, name in enumerate(available_tree_models):
        estimator = results[name]['estimator']
        # Get feature importances
        if hasattr(estimator, 'feature_importances_'):
            importances = estimator.feature_importances_
            indices = np.argsort(importances)[::-1]
            
            axes[idx].barh(range(len(importances)), importances[indices], align='center')
            axes[idx].set_yticks(range(len(importances)))
            axes[idx].set_yticklabels([X.columns[i] for i in indices], fontsize=8)
            axes[idx].set_xlabel('Feature Importance')
            axes[idx].set_title(f'{name}\nFeature Importance')
            axes[idx].invert_yaxis()
            axes[idx].grid(True, alpha=0.3, axis='x')
    
    plt.tight_layout()
    plt.show()
    
    # Print feature importance rankings
    for name in available_tree_models:
        estimator = results[name]['estimator']
        if hasattr(estimator, 'feature_importances_'):
            importances = estimator.feature_importances_
            indices = np.argsort(importances)[::-1]
            print(f"\n{name} - Feature Importance Ranking:")
            for i, idx in enumerate(indices):
                print(f"  {i+1}. {X.columns[idx]}: {importances[idx]:.4f}")
else:
    print("No tree-based models available for feature importance analysis.")

### 6.6 Learning Curves Analysis

In [ ]:
# Learning curves for all models
from sklearn.model_selection import learning_curve

# Sort models by R2 score
all_models = sorted(results.keys(), key=lambda n: results[n]['metrics']['R2'], reverse=True)

# Create grid layout for all models
n_models = len(all_models)
n_cols = 3
n_rows = (n_models + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 5*n_rows))
axes = axes.flatten() if n_models > 1 else [axes]

for idx, name in enumerate(all_models):
    model = results[name]['estimator']
    
    # Calculate learning curve
    train_sizes, train_scores, val_scores = learning_curve(
        model, X_train, y_train, cv=5, 
        train_sizes=np.linspace(0.1, 1.0, 10),
        scoring='r2', n_jobs=-1
    )
    
    train_mean = np.mean(train_scores, axis=1)
    train_std = np.std(train_scores, axis=1)
    val_mean = np.mean(val_scores, axis=1)
    val_std = np.std(val_scores, axis=1)
    
    axes[idx].plot(train_sizes, train_mean, 'o-', label='Training score', linewidth=2, color='blue')
    axes[idx].fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.15, color='blue')
    axes[idx].plot(train_sizes, val_mean, 'o-', label='Cross-validation score', linewidth=2, color='orange')
    axes[idx].fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.15, color='orange')
    
    axes[idx].set_xlabel('Training Set Size', fontsize=10)
    axes[idx].set_ylabel('R² Score', fontsize=10)
    axes[idx].set_title(f'Learning Curve - {name}\nR²={results[name]["metrics"]["R2"]:.4f}', fontsize=11)
    axes[idx].legend(loc='lower right', fontsize=9)
    axes[idx].grid(True, alpha=0.3)
    axes[idx].set_ylim([0.80, 1.005])

# Hide extra subplots
for idx in range(n_models, len(axes)):
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

print(f"\nLearning curves generated for all {n_models} models.")
print("Models ordered by R² score (best to worst).")

### 6.3 Comparison of Top 3 Models

## 7. Short discussion
Summarize model comparisons and reasoning about best performer and potential improvements (feature engineering, hyperparameter tuning, ensembling).

In [ ]:
# Summary and Discussion
print('='*80)
print('AIRFOIL REGRESSION - SUMMARY AND DISCUSSION')
print('='*80)

# Get best model
best_name = max(results.keys(), key=lambda n: results[n]['metrics']['R2'])
best = results[best_name]

print(f"\n1. BEST PERFORMING MODEL: {best_name}")
print(f"   - R² Score: {best['metrics']['R2']:.4f}")
print(f"   - RMSE: {best['metrics']['RMSE']:.4f}")
print(f"   - MAE: {best['metrics']['MAE']:.4f}")
if 'best_params' in best:
    print(f"   - Best Parameters: {best['best_params']}")

print("\n2. MODEL COMPARISON:")
for name, res in sorted(results.items(), key=lambda x: x[1]['metrics']['R2'], reverse=True)[:5]:
    print(f"   {name:20s}: R²={res['metrics']['R2']:.4f}, RMSE={res['metrics']['RMSE']:.4f}")

print("\n3. KEY INSIGHTS:")
print("   - Tree-based models (Random Forest, Gradient Boosting) generally perform well")
print("   - Neural networks (MLP) can achieve competitive results with proper tuning")
print("   - Linear models provide good baselines but may underfit complex relationships")
print("   - SVR can be powerful but requires careful hyperparameter tuning")

print("\n4. POTENTIAL IMPROVEMENTS:")
print("   - Feature engineering: polynomial features, interaction terms")
print("   - More extensive hyperparameter tuning (e.g., RandomizedSearchCV)")
print("   - Ensemble methods: stacking or blending multiple models")
print("   - Feature selection to reduce dimensionality and improve generalization")
print("   - Advanced models: XGBoost, LightGBM, CatBoost")

print("\n5. CROSS-VALIDATION ANALYSIS:")
if 'cv_mean' in best:
    print(f"   - Best model CV R²: {best['cv_mean']:.4f} (±{best['cv_std']:.4f})")
    print(f"   - Test R²: {best['metrics']['R2']:.4f}")
    diff = abs(best['cv_mean'] - best['metrics']['R2'])
    if diff < 0.05:
        print(f"   - Model shows good generalization (CV-Test difference: {diff:.4f})")
    else:
        print(f"   - Model may be overfitting (CV-Test difference: {diff:.4f})")

print('='*80)